# معمل التجربة: دمج FAISS مع سلاسل LCEL والبث الحي

في المعمل ده هنربط فهرس FAISS السريع مع نموذج Groq باستخدام مسار LCEL متكامل ومباشر.

### 1. إعداد الفهرس وسلسلة LCEL
يلا نركب السلسلة:

In [1]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

data = [
    Document(page_content="قواعد البيانات السحابية تقدم توافرية عالية ونسخ احتياطي تلقائي."),
    Document(page_content="فهارس المتجهات المحلية مثل FAISS لا تكلف أي مصاريف سحابية وتعمل دون اتصال بالإنترنت."),
]

vector_store = FAISS.from_documents(data, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 1})
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

prompt = ChatPromptTemplate.from_template("""أجب باختصار بالاعتماد على السياق:
السياق: {context}
السؤال: {question}
الإجابة:""")

chain = (
    {"context": retriever | (lambda docs: "\n".join(d.page_content for d in docs)), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("سلسلة FAISS LCEL جاهزة!")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_21728\2150085359.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

سلسلة FAISS LCEL جاهزة!


### 2. تجربة السؤال والبث
تعال نسأل ونشوف الإجابة:

In [2]:
for chunk in chain.stream("ما هي الميزة الأساسية لفهرس FAISS المحلي؟"):
    print(chunk, end="", flush=True)
print("\n")

المي

زة

 الأساسية

:

 لا

 توجد

... (11 outputs left out)

 بال

إن

ترنت

.

### 3. جرب بنفسك!
جرب تسأل عن الفهارس السحابية وشوف النموذج هيجاوب بناءً على السياق فقط ولا لأ.